---
## 📚 Sumário Geral do Módulo — Populações e Amostras (Colab-Ready)

<br>

<div align="center">

### **Projeto-guia:** _Operação Entrega Relâmpago (iFood)_

**Objetivo:** reduzir o tempo médio de entrega sem piorar NPS ou elevar custo.  
Cada aula adiciona uma “peça” estatística que destrava a próxima decisão de negócio.

</div>

<br>

<div align="center">

| # | Aula | Foco |
|:---:|:---|:---|
| **1** | **Populações e Amostras** | O que medir, por que amostrar, como comunicar incerteza |
| **2** | **Viés** | Distorções de seleção/cobertura e correções (pesos) |
| **3** | **Tipos de Amostragem** | SRS, estratos (proporcional/Neyman), sistemática |
| **4** | **Distribuições Amostrais** | Variabilidade dos estimadores; erro-padrão |
| **5** | **Teorema do Limite Central** | Por que a média “vira” Normal |
| **6** | **Tamanho Amostral** | Como definir \(n\) para média e proporção |
| **8** | **Avaliando a Amostragem** | Gates de qualidade (χ² e KS) |

</div>

<br>


---
## 🎓 Aula 1: Populações e Amostras

<br>

### **Parte 1: Introdução e Conceito**

<br>

* **O Grande “Porquê”:** medir **bem** uma parte representativa é mais barato, rápido e suficiente para decidir — se a **incerteza** for explicitada (IC e margem).
* **Fundamentos Chave**
  * **População:** universo-alvo (ex.: todos os pedidos da quinzena).
  * **Amostra:** subconjunto aleatório para estimar parâmetros.
  * **Parâmetro vs. Estatística:** \( \mu, \sigma \) (pop.) vs. \( \bar{x}, s \) (amostra).
  * **Erro-padrão (média):**  

    $$
    EP=\frac{s}{\sqrt{n}}
    $$

  * **IC95%:**  

    $$
    \bar{x}\ \pm\ z^* \cdot \frac{s}{\sqrt{n}}\quad\text{com } z^*=1{,}96
    $$

<br>


### **Parte 2: Prática e Aplicação (Projeto “Operação Entrega Relâmpago”)**

<br>

* **Visão de Negócio:** reportar \(T_{\text{médio}}\) (min) com IC95% para a diretoria.
* **Plano (antes do código):**  
  1) Carregar base; 2) SRS; 3) calcular \( \bar{x}, s, EP \) e IC; 4) checar unidades/assumptions.

<br>


In [ ]:
# Aula 1 — Estimar média e IC95% do tempo de entrega (SRS)
import numpy as np, pandas as pd, math, os

PATH = "/mnt/data/base_ifood_limpa.csv"
np.random.seed(10)

def carregar_populacao(path=PATH, col="delivery_time_min"):
    if os.path.exists(path):
        df = pd.read_csv(path)
        if col in df.columns:
            return df[col].dropna().to_numpy(float)
    # fallback sintético (assimétrico como tempos)
    return np.random.gamma(shape=2.2, scale=9.0, size=80_000)

x = carregar_populacao()

n = 600
amostra = np.random.choice(x, size=n, replace=False)

xbar = float(amostra.mean())
s = float(amostra.std(ddof=1))
ep = s / math.sqrt(n)
z = 1.96
lo, hi = xbar - z*ep, xbar + z*ep

print(f"Média (n={n}): {xbar:.2f} min")
print(f"EP: {ep:.2f} min | IC95%: [{lo:.2f}, {hi:.2f}] min")


In [ ]:
# Pratique seu código aqui!


* **Análise Detalhada do Código**

<br>

#### **Lista de Fórmulas (com parâmetros explicados)**
1) **Média amostral**  

   $$
   \bar{x}=\frac{1}{n}\sum_{i=1}^{n}x_i
   $$

   *Parâmetros:* \(x_i\) (observações), \(n\) (tamanho da amostra).

2) **Desvio-padrão amostral**  

   $$
   s=\sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2}
   $$

   *Parâmetros:* \(n-1\) (graus de liberdade; `ddof=1` no código).

3) **Erro-padrão**  

   $$
   EP=\frac{s}{\sqrt{n}}
   $$

   *Parâmetros:* \(s\) (disp. amostral), \(n\) (tamanho).

4) **IC95%**  

   $$
   \bar{x}\ \pm\ z^* \cdot EP
   $$

   *Parâmetros:* \(z^*\) (quantil da Normal padrão; 1.96 para 95%), \(EP\) (de (3)).

<br>

#### **Linhas de código (fórmula ↔ código, parâmetros e “onde entra”)**
- `np.random.choice(x, size=n, replace=False)` → gera \( \{x_i\}_{i=1}^n \) para (1)-(3).  
  *Parâmetros notáveis:* `p` (probabilidades) — **se usado**, muda o mecanismo (vira amostragem ponderada; você então deve ajustar as fórmulas usando pesos).

- `amostra.mean()` → \( \bar{x} \) em (1).

- `amostra.std(ddof=1)` → \( s \) em (2).  
  *`ddof=1`* implementa o \(n-1\) (graus de liberdade).

- `ep = s / sqrt(n)` → \( EP \) em (3); entra **multiplicando** \(z^*\) em (4).

- `lo, hi = xbar ± 1.96*ep` → constrói o intervalo (4).  
  *Parâmetros notáveis:* para \(n\) pequeno e \( \sigma \) desconhecido, troque \(z^*\) por \(t_{n-1}\): `scipy.stats.t.ppf((1+conf)/2, df=n-1)` — **onde entra:** substitui \(z^*\) na fórmula (4).

<br>

### **📝 Nota de Rodapé para Novatos**

<br>

* **Unidade/consistência:** \( \bar{x}, s, EP \) e IC em **minutos**.  
* **TLC:** com \(n\) moderado/grande, IC com \(z\) é aceitável mesmo sem normalidade perfeita.

<br>

### **Parte 3: Verificação de Aprendizado**

1. Em SRS, `replace=False` garante:  
* A) pesos iguais  
* B) sem reposição  
* C) reposição  
* D) estratificação  
* E) conglomerados  

2. O erro-padrão da média é:  
* A) \( \sigma/\sqrt{n} \)  
* B) \( s/\sqrt{n} \)  
* C) \( s^2 \)  
* D) \( \bar{x}/n \)  
* E) \( 1.96\cdot s \)  

3. IC95% significa:  
* A) \( \mu \) tem 95% de chance de cair no intervalo  
* B) o **procedimento** cobre 95% dos cenários  
* C) 95% dos dados caem no intervalo  
* D) \( \bar{x} \) é 95% certo  
* E) nada  

4. Unidade de \( EP \):  
* A) adimensional  
* B) minutos  
* C) minutos²  
* D) %  
* E) depende de \(n\)  

5. Aumentar \(n\) por 4 reduz \(EP\) por:  
* A) \(4\times\)  
* B) \(2\times\)  
* C) \(1/4\)  
* D) \(1/2\)  
* E) \(1/\sqrt{2}\)  

6. Usamos \( s \) porque:  
* A) estética  
* B) \( \sigma \) é desconhecido  
* C) TLC exige  
* D) sempre  
* E) nunca  

7. Em dados assimétricos, usar \(z\) requer:  
* A) \(n\) pequeno  
* B) normalidade perfeita  
* C) \(n\) moderado/grande (TLC)  
* D) pesos  
* E) nada  

8. `ddof=1` implica divisor:  
* A) \(n\)  
* B) \(n+1\)  
* C) \(n-1\)  
* D) \(n^2\)  
* E) \(1\)  

9. População vs. amostra:  
* A) ambos subsets  
* B) amostra ⊂ população  
* C) população ⊂ amostra  
* D) iguais  
* E) disjuntos  

10. KPIs e unidades devem:  
* A) ser omitidos  
* B) aparecer só no fim  
* C) acompanhar todos os números  
* D) variar  
* E) ser percentuais  

<br>

### **🔑 Gabarito das Perguntas de Verificação**

1. **B** 2. **B** 3. **B** 4. **B** 5. **D** 6. **B** 7. **C** 8. **C** 9. **B** 10. **C**

<br>


---
## 🎓 Aula 2: Viés

<br>

### **Parte 1: Introdução e Conceito**

<br>

* **O Grande “Porquê”:** _viés_ = **erro sistemático**; não some com mais \(n\).  
* **Tipos comuns:** seleção/cobertura, não resposta, auto-seleção, geográfico, temporal, socioeconômico.  
* **Correções:** melhor **desenho**; **pós-estratificação** com pesos; **reamostragem**.

<br>


### **Parte 2: Prática e Aplicação (Projeto “Operação Entrega Relâmpago”)**

<br>

* **Visão:** amostra colhida só em capitais (SP/RJ) → tempos parecem menores. É real ou viés?  
* **Plano:** comparar SRS vs. conveniência; aplicar pesos  

  $$
  w_h=\frac{P_h}{\hat{p}_h}
  $$

  e média ponderada  

  $$
  \bar{x}_w=\frac{\sum w_i x_i}{\sum w_i}.
  $$

<br>


In [ ]:
# Aula 2 — Diagnóstico de viés e correção por pós-estratificação
import numpy as np, pandas as pd
np.random.seed(11)

N = 50_000
df = pd.DataFrame({
    "city": np.random.choice(["SP","RJ","BH","SSA","POA"], p=[0.45,0.20,0.15,0.12,0.08], size=N)
})
mu = {"SP":40,"RJ":42,"BH":48,"SSA":50,"POA":47}
sigma = {"SP":10,"RJ":11,"BH":12,"SSA":13,"POA":11}
df["delivery_time_min"] = df["city"].map(mu) + np.random.normal(0,1,size=N)*df["city"].map(sigma)

srs = df.sample(n=1200, random_state=12)
xbar_srs = srs["delivery_time_min"].mean()

conv = df[df["city"].isin(["SP","RJ"])].sample(n=1200, random_state=13)
xbar_conv = conv["delivery_time_min"].mean()

pop_prop = df["city"].value_counts(normalize=True).sort_index()
am_prop  = conv["city"].value_counts(normalize=True).reindex(pop_prop.index).fillna(0)
w = (pop_prop / am_prop).replace([np.inf, -np.inf], np.nan).fillna(0)

xbar_weighted = (conv["delivery_time_min"] * conv["city"].map(w)).sum() / conv["city"].map(w).sum()

print(f"Média SRS    : {xbar_srs:.2f} min")
print(f"Média Conv   : {xbar_conv:.2f} min (possível viés)")
print(f"Média Corrig.: {xbar_weighted:.2f} min (pós-estratificada)")


In [ ]:
# Pratique seu código aqui!


* **Análise Detalhada do Código**

<br>

#### **Fórmulas (com parâmetros)**
1) **Peso de pós-estrato**  

   $$
   w_h=\frac{P_h}{\hat{p}_h}
   $$

   *Parâmetros:* \(P_h\) (proporção populacional do estrato \(h\)); \(\hat{p}_h\) (proporção **na amostra**).

2) **Média ponderada**  

   $$
   \bar{x}_w=\frac{\sum_i w_i x_i}{\sum_i w_i}
   $$

   *Parâmetros:* \(w_i\) (peso do item \(i\)), \(x_i\) (observação).

<br>

#### **Linhas (fórmula ↔ código, parâmetros e “onde entra”)**
- `value_counts(normalize=True)` → calcula \(P_h\) e \(\hat{p}_h\).  
  *Parâmetro notável:* `normalize=True` retorna proporções em vez de contagens.

- `w = (pop_prop / am_prop)` → (1).  
  *Cuidados:* se \(\hat{p}_h=0\) → \(w_h\to\infty\) (estrato ausente). **Onde entra:** exige **reamostrar** ou **agrupar** estratos; em produção, aplique truncagem de pesos (ex.: \([0.3, 3]\)).

- `(conv["delivery_time_min"] * conv["city"].map(w)).sum() / conv["city"].map(w).sum()` → (2).  
  *Onde entra:* substitui a média simples quando a amostra foi coletada com desbalanceamento entre estratos.

<br>

### **📝 Nota de Rodapé para Novatos**

<br>

* **Mais \(n\) não corrige viés** — só reduz variância do erro.  
* **Pós-estratificação** “faz” a amostra parecer a população ao reponderar os estratos.

<br>

### **Parte 3: Verificação de Aprendizado**

1. Viés de cobertura ocorre quando:  
* A) sorteio sem reposição  
* B) **parte da população fica fora do frame**  
* C) \(n\) pequeno  
* D) \(EP\) alto  
* E) \(p\) desconhecido  

2. A correção via pesos usa:  
* A) \( w_h=\hat{p}_h/P_h \)  
* B) \( w_h=P_h/\hat{p}_h \)  
* C) \( w_h=P_h\cdot\hat{p}_h \)  
* D) \( w_h=1 \)  
* E) \( w_h=0 \)  

3. Aumentar \(n\) numa amostra enviesada:  
* A) corrige viés  
* B) piora sempre  
* C) **não corrige**  
* D) torna IC inválido  
* E) muda unidade  

4. Pós-estratificação ajusta:  
* A) a população  
* B) a **estimativa** (repondera estratos)  
* C) variância populacional  
* D) tamanho populacional  
* E) resíduos  

5. Se \(\hat{p}_h=0\), então \(w_h\) é:  
* A) **infinito (problema)**  
* B) zero  
* C) um  
* D) negativo  
* E) \(P_h\)  

6. Estratégia quando \(\hat{p}_h=0\):  
* A) aceitar  
* B) truncar  
* C) **reamostrar** ou **agrupar** estratos  
* D) aumentar \(z\)  
* E) usar \(t\)  

7. Pesos grandes:  
* A) reduzem variância  
* B) **podem aumentar** variância  
* C) não mudam nada  
* D) garantem ICs menores  
* E) anulam viés sempre  

8. Viés é:  
* A) erro aleatório  
* B) **erro sistemático**  
* C) ruído  
* D) mediana  
* E) moda  

9. O uso de `p` em `np.random.choice(..., p=...)`:  
* A) mantém SRS  
* B) **cria amostragem ponderada**  
* C) ignora pesos  
* D) impede IC  
* E) define \(z\)  

10. Ajustar por `city×price_range` cria:  
* A) **estratos compostos**  
* B) conglomerados  
* C) systematic bias  
* D) overfitting sempre  
* E) nada  

<br>

### **🔑 Gabarito das Perguntas de Verificação**

1. **B** 2. **B** 3. **C** 4. **B** 5. **A** 6. **C** 7. **B** 8. **B** 9. **B** 10. **A**

<br>


---
## 🎓 Aula 3: Tipos de Amostragem

<br>

### **Parte 1: Introdução e Conceito**

<br>

* **O Grande “Porquê”:** o **como** amostrar controla custo e precisão.  
* **Métodos:**  
  * **SRS** (simples);  
  * **Estratificada** — alocação proporcional  

    $$
    n_h=n\cdot \frac{N_h}{N}
    $$

    e **Neyman**  

    $$
    n_h=n\cdot\frac{N_h S_h}{\sum_j N_j S_j}
    $$

  * **Sistemática:** 1 a cada \(k\) após arranque aleatório;  
  * **Conglomerados:** sorteia clusters (design effect ↑).

<br>


### **Parte 2: Prática e Aplicação**

<br>

* **Visão:** representar `city` eficientemente.  
* **Plano:** comparar estratificada proporcional vs. Neyman (quando \(S_h\) difere).

<br>


In [ ]:
# Aula 3 — Amostragem estratificada: proporcional vs. Neyman
import numpy as np, pandas as pd
np.random.seed(15)

N = 60_000
df = pd.DataFrame({
    "city": np.random.choice(["SP","RJ","BH","SSA","POA"], p=[0.45,0.20,0.15,0.12,0.08], size=N)
})
mu = {"SP":40,"RJ":42,"BH":48,"SSA":50,"POA":47}
sigma = {"SP":9,"RJ":11,"BH":13,"SSA":14,"POA":12}
df["delivery_time_min"] = df["city"].map(mu) + np.random.normal(0,1,size=N)*df["city"].map(sigma)

n = 800
prop = df["city"].value_counts(normalize=True).sort_index()
Sh = df.groupby("city")["delivery_time_min"].std(ddof=1).sort_index()

n_prop = (prop * n).round().astype(int)
weights = (df["city"].value_counts().sort_index() * Sh)
n_neyman = (n * (weights/weights.sum())).round().astype(int)

def amostra_por_alocacao(df, n_by_cat, col="city", seed=15):
    partes=[]
    for cat, nn in n_by_cat.items():
        partes.append(df[df[col]==cat].sample(n=int(nn), replace=False, random_state=seed))
    return pd.concat(partes, ignore_index=True)

a_prop   = amostra_por_alocacao(df, n_prop)
a_neyman = amostra_por_alocacao(df, n_neyman)

out = pd.DataFrame({
    "Pop%": (prop*100).round(2),
    "Prop%": (a_prop["city"].value_counts(normalize=True).sort_index()*100).round(2),
    "Neyman%": (a_neyman["city"].value_counts(normalize=True).sort_index()*100).round(2)
})
print(out)


In [ ]:
# Pratique seu código aqui!


* **Análise Detalhada do Código**

<br>

#### **Fórmulas (com parâmetros)**
1) **Proporcional:**  

   $$
   n_h=n\cdot\frac{N_h}{N}
   $$

   *Parâmetros:* \(N_h\) (tamanho do estrato), \(N\) (pop. total), \(n\) (amostra total).

2) **Neyman:**  

   $$
   n_h=n\cdot\frac{N_h S_h}{\sum_j N_j S_j}
   $$

   *Parâmetros:* \(S_h\) (desvio no estrato), \(N_h\) (peso do estrato).

<br>

#### **Linhas (fórmula ↔ código)**
- `prop = value_counts(normalize=True)` → \(N_h/N\) para (1).  
- `groupby.std(ddof=1)` → \(S_h\) para (2).  
- `weights = N_h * S_h` e `weights/weights.sum()` → razão em (2).  
- **Onde entra arredondamento:** `round()` pode fazer \(\sum_h n_h \neq n\). Redistribua sobras e imponha piso \(n_h^{\min}\) quando relatórios por estrato forem obrigatórios.

<br>

### **📝 Nota de Rodapé para Novatos**

<br>

* **Estrato:** grupo relativamente homogêneo (categoria).  
* **Intuição de Neyman:** amostrar **mais** onde há **mais variação** e **mais população**.

<br>

### **Parte 3: Verificação de Aprendizado**

1. Em alocação proporcional, \( n_h \) depende de:  
* A) \( S_h \)  
* B) \( N_h \)  
* C) \( z \)  
* D) \( t \)  
* E) \( \mu \)  

2. Neyman prioriza estratos com:  
* A) menor \( S_h \)  
* B) **maior \( S_h \) e \( N_h \)**  
* C) maior \( \mu \)  
* D) menor \( N_h \)  
* E) fixo  

3. Conglomerados tendem a:  
* A) reduzir deff  
* B) **aumentar deff**  
* C) não afetar  
* D) invalidar IC  
* E) nada  

4. Sistemática exige:  
* A) **ordem aleatória (sem periodicidade)**  
* B) ordem crescente  
* C) pesos  
* D) \( k=1 \)  
* E) conglomerados  

5. Estratificar ajuda quando:  
* A) população homogênea  
* B) **heterogênea por subgrupos**  
* C) \( n=1 \)  
* D) piora sempre  
* E) nunca  

6. Se \( S_h \) são similares:  
* A) **Neyman ≈ proporcional**  
* B) Neyman difere muito  
* C) usar conglomerados  
* D) usar sistemática  
* E) nada  

7. Em sistemática, o “arranque” aleatório:  
* A) é opcional  
* B) **evita viés** com periodicidade  
* C) aumenta viés  
* D) zera deff  
* E) define \( S_h \)  

8. Piso \( n_h^{\min} \) serve para:  
* A) estética  
* B) **garantir relatório por estrato**  
* C) reduzir viés  
* D) mudar \( z \)  
* E) alterar unidade  

9. SRS pode falhar quando:  
* A) todos os estratos são iguais  
* B) **há subgrupos raros**  
* C) \( N \) é grande  
* D) \( n \) é grande  
* E) \( z \) é alto  

10. A meta da estratificação é:  
* A) subir custo  
* B) **reduzir variância** de estimadores  
* C) trocar KPI  
* D) mudar unidade  
* E) fixar \( z \)  

<br>

### **🔑 Gabarito das Perguntas de Verificação**

1. **B** 2. **B** 3. **B** 4. **A** 5. **B** 6. **A** 7. **B** 8. **B** 9. **B** 10. **B**

<br>


---
## 🎓 Aula 4: Distribuições Amostrais

<br>

### **Parte 1: Introdução e Conceito**

<br>

* **O Grande “Porquê”:** entender o **comportamento** do estimador ao repetir a amostragem para construir margens confiáveis.  
* **Chaves:**  

  $$
  \bar{X} \approx \mathcal{N}\!\Big(\mu,\ \frac{\sigma^2}{n}\Big),\qquad
  EP=\frac{\sigma}{\sqrt{n}}
  $$

<br>


### **Parte 2: Prática e Aplicação**

<br>

* **Visão:** prever a **margem** ao escolher \(n\).  
* **Plano:** simular muitas amostras de população assimétrica e observar \(EP\).

<br>


In [ ]:
# Aula 4 — Simulando a distribuição amostral da média
import numpy as np
np.random.seed(20)

pop = np.random.exponential(scale=1.0, size=600_000)  # média=1.0

def medias_amostrais(n=30, B=4000):
    return np.array([np.mean(np.random.choice(pop, size=n, replace=False)) for _ in range(B)])

for n in [30, 100, 400]:
    meds = medias_amostrais(n)
    print(f"n={n:3d} | média das médias={meds.mean():.3f} | EP≈{meds.std(ddof=1):.4f}")


In [ ]:
# Pratique seu código aqui!


* **Análise Detalhada do Código**

<br>

#### **Fórmulas (com parâmetros)**
1) \( EP=\sigma/\sqrt{n} \) — decai com \(1/\sqrt{n}\).  
2) \( \bar{X}\sim \mathcal{N}(\mu,\sigma^2/n) \) (aprox. via TLC).

<br>

#### **Linhas (fórmula ↔ código)**
- `np.random.exponential(...)` → base não Normal para testar robustez do TLC.  
- `np.random.choice(...).mean()` → realizações de \( \bar{X} \).  
- `meds.std(ddof=1)` → estima \(EP\) empírico (desvio de \( \bar{X} \)).

* **Parâmetros notáveis:** \(n\) entra no **denominador**; \(B\) (nº de simulações) só melhora a **precisão** de estimar \(EP\), não muda \(EP\) verdadeiro.

<br>

### **📝 Nota de Rodapé para Novatos**

<br>

* A média de amostras “normaliza” formas originais pela soma (TLC).  
* Reduzir \(EP\) pela metade exige **quadruplicar** \(n\).

<br>

### **Parte 3: Verificação de Aprendizado**

1. A distribuição amostral descreve:  
* A) dados brutos  
* B) **o estimador ao repetir amostras**  
* C) a população  
* D) resíduos  
* E) nada  

2. \( EP \) da média cai com:  
* A) \(1/n\)  
* B) **\(1/\sqrt{n}\)**  
* C) \(n\)  
* D) \(n^2\)  
* E) \(z\)  

3. Em população Exponencial, a média das amostras:  
* A) é Exponencial  
* B) **é Normal aprox.**  
* C) é Uniforme  
* D) é \(t\)  
* E) é Pareto  

4. Aumentar \(B\) muda:  
* A) \( \mu \)  
* B) \( EP \) populacional  
* C) **precisão da estimativa** de \( EP \)  
* D) nada  
* E) unidade  

5. Se \(n\) cresce, a forma de \( \bar{X} \) tende a:  
* A) Exponencial  
* B) **Normal**  
* C) Uniforme  
* D) Cauchy  
* E) Beta  

6. Diferença entre \(s\) e \( \sigma \) afeta:  
* A) **numerador de \(EP\)**  
* B) denominador de \(EP\)  
* C) \(z\)  
* D) \(n\)  
* E) unidade  

7. O desvio de `meds` (muitas amostras) estima:  
* A) \(s\)  
* B) \( \sigma \)  
* C) **\( EP \)**  
* D) \(z\)  
* E) \(t\)  

8. Para reduzir \(EP\) \( \times 0.5\), \(n\) precisa:  
* A) dobrar  
* B) **quadruplicar**  
* C) reduzir à metade  
* D) manter  
* E) \( \times 1.5\)  

9. A razão para “normalizar” a média é:  
* A) arredondamento  
* B) **somas suavizam variação (TLC)**  
* C) outliers  
* D) pesos  
* E) amostras pequenas  

10. Distribuição amostral é central para:  
* A) gráficos  
* B) **EP e IC**  
* C) ETL  
* D) clusterização  
* E) PCA  

<br>

### **🔑 Gabarito das Perguntas de Verificação**

1. **B** 2. **B** 3. **B** 4. **C** 5. **B** 6. **A** 7. **C** 8. **B** 9. **B** 10. **B**

<br>


---
## 🎓 Aula 5: Teorema do Limite Central

<br>

### **Parte 1: Introdução e Conceito**

<br>

* **O Grande “Porquê”:** legitima ICs/testes baseados em Normal para \( \bar{X} \) em muitos cenários.  
* **Declaração (forma padronizada):**  

  $$
  Z=\frac{\bar{X}-\mu}{\sigma/\sqrt{n}} \xrightarrow{d} \mathcal{N}(0,1)
  $$

  *Parâmetros:* \( \mu,\sigma \) (pop.), \(n\) (tamanho).

<br>


### **Parte 2: Prática e Aplicação**

<br>

* **Visão:** justificar à diretoria uso de \(z\) com \(n\ge 50\).  
* **Plano:** simular \(Z\) e comparar fração \(|Z|>1.96\) (~5%).

<br>


In [ ]:
# Aula 5 — Verificando a aproximação Normal via TLC
import numpy as np
np.random.seed(33)

pop = np.random.exponential(scale=1.2, size=500_000)  # média=1.2

def zscores_medias(n=50, B=4000):
    zs=[]
    mu = pop.mean(); sigma = pop.std(ddof=1)
    for _ in range(B):
        s = np.random.choice(pop, size=n, replace=False)
        z = (s.mean() - mu) / (sigma/np.sqrt(n))
        zs.append(z)
    zs = np.array(zs)
    return zs, np.mean(np.abs(zs)>1.96)

for n in [20, 50, 200]:
    zs, frac = zscores_medias(n)
    print(f"n={n:3d} | frac(|Z|>1.96)≈{frac:.3f} (esperado ~0.05)")


In [ ]:
# Pratique seu código aqui!


* **Análise Detalhada do Código**

<br>

#### **Fórmulas (com parâmetros)**
1)  

   $$
   Z=\frac{\bar{X}-\mu}{\sigma/\sqrt{n}}
   $$

   *Parâmetros:* \(\bar{X}\) (média amostral), \(\mu\) (média pop.), \(\sigma\) (desvio pop.), \(n\) (tamanho).

2) **Cobertura 95%:**  

   $$
   \Pr(|Z|>1.96)\approx 0.05
   $$

<br>

#### **Linhas (fórmula ↔ código)**
- `pop.mean(), pop.std(ddof=1)` → estimam \(\mu,\sigma\).  
- `(s.mean()-mu)/(sigma/np.sqrt(n))` → \(Z\) (1).  
- `mean(abs(zs)>1.96)` → aproxima (2).

* **Parâmetros notáveis:** i.i.d. e variância finita — com caudas muito pesadas, a aproximação pode falhar.

<br>

### **📝 Nota de Rodapé para Novatos**

<br>

* **Usar \(t\)** quando \(n\) pequeno e \( \sigma \) desconhecido.  
* **Convergência em distribuição:** as CDFs se aproximam à Normal conforme \(n\) cresce.

<br>

### **Parte 3: Verificação de Aprendizado**

1. TLC aplica-se a:  
* A) **média com variância finita**  
* B) máximo  
* C) mediana  
* D) moda  
* E) outliers  

2. Se \(n\) aumenta, \( \Pr(|Z|>1.96) \) tende a:  
* A) 0  
* B) **0.05**  
* C) 0.5  
* D) 1  
* E) 0.95  

3. Premissas-chave:  
* A) normalidade perfeita  
* B) **i.i.d. e variância finita**  
* C) Pareto  
* D) Uniforme  
* E) Exponencial  

4. Com variância infinita, TLC:  
* A) igual  
* B) **pode falhar**  
* C) melhora  
* D) não importa  
* E) garante Normal  

5. Usar \(z\) em vez de \(t\) com \(n\) pequeno:  
* A) sempre ok  
* B) conservador  
* C) **arriscado**  
* D) igual  
* E) proibido  

6. O denominador de \(Z\) é:  
* A) \( \sigma \)  
* B) **\( \sigma/\sqrt{n} \)**  
* C) \( s \)  
* D) \( s/\sqrt{n} \)  
* E) \( n \)  

7. A melhoria com \(n\) reflete:  
* A) lei dos grandes números  
* B) **convergência em distribuição de \( \bar{X} \)**  
* C) teorema de Slutsky  
* D) invariância de escala  
* E) bootstrap  

8. Se \( \sigma \) é desconhecido, substituímos por:  
* A) **\( s \) e usamos \( t \)**  
* B) \( s \) e usamos \( z \)  
* C) \( \bar{x} \)  
* D) \( n \)  
* E) \( 0 \)  

9. Amostragem dependente (clusters sem ajuste) tende a:  
* A) reduzir variância  
* B) **inflar variância efetiva**  
* C) não afetar  
* D) zerar \(Z\)  
* E) trocar \( \mu \)  

10. Para \(n=20\) e base assimétrica, é mais seguro:  
* A) usar \( z \) direto  
* B) **usar \( t \) e/ou aumentar \( n \)**  
* C) ignorar \( EP \)  
* D) usar mediana  
* E) usar média truncada sem IC  

<br>

### **🔑 Gabarito das Perguntas de Verificação**

1. **A** 2. **B** 3. **B** 4. **B** 5. **C** 6. **B** 7. **B** 8. **A** 9. **B** 10. **B**

<br>


---
## 🎓 Aula 6: Tamanho Amostral

<br>

### **Parte 1: Introdução e Conceito**

<br>

* **O Grande “Porquê”:** defina \(n\) **antes** de coletar — economiza e dá previsibilidade de **margem** e **poder**.  
* **Fórmulas chave**
  * **Média:**  

    $$
    n=\left(\frac{z\cdot\sigma}{E}\right)^2
    $$

  * **Proporção:**  

    $$
    n=\frac{z^2\,p(1-p)}{E^2}
    $$

  * **Correção finita:**  

    $$
    n_{\text{corr}}=\frac{N n_0}{N+n_0-1}
    $$

  *Parâmetros gerais:* \(z\) (nível de confiança), \(\sigma\) (disp. pop.), \(p\) (proporção alvo), \(E\) (margem desejada), \(N\) (tamanho pop.).

<br>


### **Parte 2: Prática e Aplicação**

<br>

* **Visão:** prometer IC95% de \( \pm 2 \) min (histórico \( \sigma\approx 15{,}5 \)).  
* **Plano:** utilitários para média, proporção, correção finita; explicar cada parâmetro.

<br>


In [ ]:
# Aula 6 — Utilitários de tamanho amostral
import math

def n_media(sigma: float, margem: float, conf: float=0.95) -> int:
    zmap={0.90:1.645,0.95:1.960,0.99:2.576}; z=zmap.get(conf,1.960)
    return math.ceil((z*sigma/margem)**2)

def n_proporcao(p: float, margem: float, conf: float=0.95) -> int:
    zmap={0.90:1.645,0.95:1.960,0.99:2.576}; z=zmap.get(conf,1.960)
    return math.ceil((z**2 * p*(1-p))/margem**2)

def n_finita(n0: int, N: int) -> int:
    return math.ceil((N*n0)/(N+n0-1))

print("Média: σ≈15.5, 95%, ±2 min  → n≈", n_media(15.5, 2.0))
print("Prop.: p≈0.05, 95%, ±1 p.p. → n≈", n_proporcao(0.05, 0.01))
print("Correção finita (N=50k)     → n_corr≈", n_finita(n_proporcao(0.05,0.01), 50_000))


In [ ]:
# Pratique seu código aqui!


* **Análise Detalhada do Código**

<br>

#### **Fórmulas (com parâmetros)**
1)  

   $$
   n=\left(\frac{z\sigma}{E}\right)^2
   $$

   *Parâmetros:* \(z\) (confiança), \(\sigma\) (desvio), \(E\) (margem).  
   **Onde entra no código:** `math.ceil((z*sigma/margem)**2)`.

2)  

   $$
   n=\frac{z^2 p(1-p)}{E^2}
   $$

   *Parâmetros:* \(p\) (proporção alvo), \(E\) (margem em **pontos percentuais**).  
   **Onde entra:** `math.ceil((z**2 * p*(1-p))/margem**2)`.  
   *Nota:* se \(p\) **desconhecido**, usar \(p=0.5\) (pior caso, \(n\) maior).

3)  

   $$
   n_{\text{corr}}=\frac{N n_0}{N+n_0-1}
   $$

   *Parâmetros:* \(N\) (população), \(n_0\) (amostra inicial).  
   **Onde entra:** `math.ceil((N*n0)/(N+n0-1))`.

* **Parâmetros notáveis:** aumentar confiança (90%→99%) sobe \(z\) e **aumenta \(n\)**; **metade da margem → ~4× \(n\)**.

<br>

### **📝 Nota de Rodapé para Novatos**

<br>

* **Consistência dimensional:** \(E\) e \(\sigma\) na **mesma unidade** (minutos).  
* Em proporção, \(E\) é **absoluto** (pontos percentuais).

<br>

### **Parte 3: Verificação de Aprendizado**

1. Dobrar \(n\) reduz a margem em:  
* A) \(1/2\)  
* B) **\(1/\sqrt{2}\)**  
* C) \(1/4\)  
* D) \(2\)  
* E) igual  

2. Para proporção desconhecida, escolha \(p\):  
* A) 0.0  
* B) 1.0  
* C) **0.5**  
* D) 0.05  
* E) 0.95  

3. Correção finita é relevante quando:  
* A) **\( n/N \) é grande**  
* B) \( n/N \) é pequeno  
* C) \( N \to \infty \)  
* D) \( p=0.5 \)  
* E) nunca  

4. Para IC95% \( \pm 2 \) min e \( \sigma=16 \), \(n\) cresce se:  
* A) **\(E\) cai**  
* B) \(E\) sobe  
* C) \(z\) cai  
* D) \( \sigma \) cai  
* E) nada  

5. Em A/B de conversão, \(n\) depende de:  
* A) **\(p, E, z\)**  
* B) \( \mu \)  
* C) \( S_h \)  
* D) \( k \)  
* E) \( ddof \)  

6. Se \( z \) muda 1.96 → 2.576:  
* A) \(n\) diminui  
* B) **\(n\) aumenta**  
* C) \(n\) igual  
* D) 0  
* E) independe  

7. Se \(E\) é em **minutos**, então:  
* A) \(E\) deve ser em %  
* B) **\(E\) e \( \sigma \) na mesma unidade**  
* C) \(E\) adimensional  
* D) usar p.p.  
* E) usar variância  

8. “Pontos percentuais” significam:  
* A) diferença em %  
* B) **diferença absoluta (5%→6% = +1 p.p.)**  
* C) variância  
* D) EP  
* E) \(z\)  

9. Para reduzir \(E\) pela metade, \(n\) precisa:  
* A) dobrar  
* B) **quadruplicar**  
* C) reduzir à metade  
* D) manter  
* E) +50%  

10. \( n_{\text{corr}} \) é:  
* A) > \( n_0 \) sempre  
* B) **≤ \( n_0 \)**  
* C) = \( n_0 \)  
* D) aleatório  
* E) indefinido  

<br>

### **🔑 Gabarito das Perguntas de Verificação**

1. **B** 2. **C** 3. **A** 4. **A** 5. **A** 6. **B** 7. **B** 8. **B** 9. **B** 10. **B**

<br>


---
## 🎓 Aula 8: Avaliando a Amostragem em Python

<br>

### **Parte 1: Introdução e Conceito**

<br>

* **O Grande “Porquê”:** amostra **só entra** no relatório se “parecer” a população — **gate** de qualidade.  
* **Ferramentas:**  
  * **χ² (categóricas):**  

    $$
    \chi^2=\sum_c \frac{(O_c-E_c)^2}{E_c}
    $$

  * **KS (contínuas):**  

    $$
    D=\sup_x \big|F_1(x)-F_2(x)\big|
    $$

<br>


### **Parte 2: Prática e Aplicação**

<br>

* **Visão:** auditar `city`, `price_range` (χ²) e `delivery_time_min` (KS).  
* **Plano:** escalar observados para mesmo total; usar KS bicaudal.

<br>


In [ ]:
# Aula 8 — Auditoria estatística: χ² (categórico) e KS (contínuo)
import numpy as np, pandas as pd
from scipy.stats import chisquare, ks_2samp

np.random.seed(77)
N = 80_000
df_pop = pd.DataFrame({
    "city": np.random.choice(["SP","RJ","BH","SSA","POA"], p=[0.45,0.20,0.15,0.12,0.08], size=N),
    "price_range": np.random.choice(["CHEAPEST","CHEAP","MODERATE","EXPENSIVE","MOST_EXPENSIVE"],
                                    p=[0.70,0.12,0.11,0.05,0.02], size=N)
})
mu = {"SP":40,"RJ":42,"BH":48,"SSA":50,"POA":47}
sigma = {"SP":9,"RJ":11,"BH":13,"SSA":14,"POA":12}
df_pop["delivery_time_min"] = df_pop["city"].map(mu) + np.random.normal(0,1,size=N)*df_pop["city"].map(sigma)

# SRS candidata
am = df_pop.sample(n=1200, random_state=78)

def chi2_pop_am(pop_series, am_series):
    pop_ct = pop_series.value_counts().sort_index()
    am_ct  = am_series.value_counts().reindex(pop_ct.index).fillna(0)
    escala = pop_ct.sum()/am_ct.sum()
    stat, p = chisquare(f_obs=am_ct.values*escala, f_exp=pop_ct.values)  # modo GOF
    return stat, p

def ks_pop_am(pop_col, am_col):
    return ks_2samp(pop_col, am_col, alternative="two-sided", mode="auto")

chi_city = chi2_pop_am(df_pop["city"], am["city"])
chi_price = chi2_pop_am(df_pop["price_range"], am["price_range"])
ks_time = ks_pop_am(df_pop["delivery_time_min"], am["delivery_time_min"])

print(f"χ² city : stat={chi_city[0]:.2f}, p={chi_city[1]:.3f}")
print(f"χ² price: stat={chi_price[0]:.2f}, p={chi_price[1]:.3f}")
print(f"KS time : D={ks_time[0]:.3f}, p={ks_time[1]:.3f}")


In [ ]:
# Pratique seu código aqui!


* **Análise Detalhada do Código**

<br>

#### **Fórmulas (com parâmetros)**
1)  

   $$
   \chi^2=\sum_c \frac{(O_c-E_c)^2}{E_c}
   $$

   *Parâmetros:* \(O_c\) (observados escalados), \(E_c\) (esperados pop.).

2)  

   $$
   D=\sup_x|F_1(x)-F_2(x)|
   $$

   *Parâmetros:* \(F_1\), \(F_2\) (CDFs empíricas).

<br>

#### **Linhas (fórmula ↔ código, parâmetros e “onde entra”)**
- `chisquare(f_obs=am*escala, f_exp=pop)` → (1); **onde entra `escala`:** garante totais iguais para aplicar χ² corretamente.  
  *Parâmetros notáveis:* categorias com \(E_c<5\) → **agrupar** antes do teste.

- `ks_2samp(..., alternative="two-sided")` → (2).  
  *Parâmetros notáveis:* transformar `np.log1p(tempo)` **antes** do KS pode aumentar sensibilidade a diferenças multiplicativas.

<br>

### **📝 Nota de Rodapé para Novatos**

<br>

* **\(p\)-valor:** compatibilidade com \(H_0\) (mesma distribuição).  
* **Fluxo prático:** reprovar → reamostrar **ou** ponderar.

<br>

### **Parte 3: Verificação de Aprendizado**

1. χ² compara:  
* A) médias  
* B) variâncias  
* C) **frequências por categoria**  
* D) medianas  
* E) máximos  

2. KS mede:  
* A) diferença de médias  
* B) **diferença máxima entre CDFs**  
* C) correlação  
* D) regressão  
* E) nada  

3. \(p\) muito pequeno sugere:  
* A) amostras iguais  
* B) **diferença estatisticamente significativa**  
* C) erro de código  
* D) Normalidade  
* E) Pareto  

4. Escalar `f_obs` em χ² serve para:  
* A) mudar \(p\)  
* B) **igualar totais**  
* C) normalizar por \(z\)  
* D) remover viés  
* E) evitar KS  

5. Após reprovar em χ², opções:  
* A) ignorar  
* B) **reamostrar**  
* C) **ponderar**  
* D) **B ou C**  
* E) encerrar  

6. Em KS, \( D \) é:  
* A) média das diferenças  
* B) **máxima** diferença ao longo de \(x\)  
* C) soma das diferenças  
* D) variância  
* E) EP  

7. Em χ², níveis com \(E_c<5\):  
* A) sem problema  
* B) **recomenda-se agrupar**  
* C) remover amostra  
* D) usar \(z\)  
* E) usar \(t\)  

8. KS é mais sensível a diferenças em:  
* A) **caudas**  
* B) picos  
* C) médias  
* D) variâncias  
* E) mediana  

9. Para variáveis contínuas com outliers, útil:  
* A) ignorar  
* B) **transformar (ex.: log) e comparar**  
* C) trocar por χ²  
* D) usar PCA  
* E) usar média truncada  

10. Gate de qualidade antes de concluir:  
* A) não necessário  
* B) **necessário (χ²/KS)**  
* C) apenas KS  
* D) apenas χ²  
* E) apenas visual  

<br>

### **🔑 Gabarito das Perguntas de Verificação**

1. **C** 2. **B** 3. **B** 4. **B** 5. **D** 6. **B** 7. **B** 8. **A** 9. **B** 10. **B**

<br>


---
## ✅ Encerramento — Pipeline de Decisão (Resumo Executivo)

<br>

1) **Defina população e KPI** (evite cobertura falha).  
2) **Desenhe a amostra** (estratos se necessário; Neyman se \(S_h\) difere).  
3) **Audite a amostra** (χ²/KS).  
4) **Reporte** \( \bar{x} \pm \) margem (95%) com **unidade** e \(n\).  
5) **A/B**: dimensione \(n\) por proporção, alinhe risco/margem antes.  
6) **Documente suposições** (i.i.d., aproximação Normal, fonte de \( \sigma \)).

<br>


---
## 🎨 Notas de Design/Didática aplicadas (o que melhoramos)

<br>

- **Espaçamento e divisores:** linhas em branco e `---` separando blocos; ` <br> ` isolado para “respiro”.  
- **Fórmulas em destaque:** uso de **$$…$$** para exibição centralizada e listas numeradas de fórmulas com **parâmetros explicados**.  
- **Tabelas centralizadas:** `div align="center"` no Sumário.  
- **Código curto + análise longa:** cada trecho vem seguido de **“Fórmulas ↔ Código”** e **“Parâmetros Notáveis (onde entra na fórmula)”**.  
- **Quizzes legíveis:** alternativas em lista com quebra de linha.  
- **Unidades/KPI:** reforçadas junto de cada métrica.  

> Dica de navegação entre aulas no Colab: use o sumário (ícone de lista) para saltar por **títulos `##`/`###`** e mantenha esta organização em futuras aulas.

<br>
